# Irrigation Training — **v2.21** (gamma-correct biomass shaping)

Separate version from v2.20's n-step stabilisation (prefix `td3_v221_*`, manifest 2.21.0).

**v2.21 = v2.20 Run A (exact n-step, which converged) + ONE change:** the biomass reward r1 switches from the increment `(x4_t - x4_{t-1})/X4_REF` to the **gamma-correct potential-shaping form** `(gamma*x4_t - x4_{t-1})/X4_REF`. Its discounted sum telescopes to exactly `gamma^T*x4_T` — a **pure terminal-yield objective, the same one MPC optimises** — with no front-loading term, and without touching gamma (so n-step stability is unchanged). This is the principled fix for the v2.20 drought seesaw.

The shaping gamma is set by the trainer to `gamma_base` (0.99) so it can't drift. Backward-compatible: the env param defaults to 1.0 (old behaviour), and the eval harness measures physical outcomes (never the reward), so the MPC comparison is unaffected — this changes only how the policy is trained.

**Before running:** push the modified `gym_env.py`, plus `configs_v221.py` and `train_v221_td3.py`; or run the *Write v2.21 files* cell. v2.20 files are untouched. ~1–1.5 hr T4.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; DRIVE_ROOT='/content/drive/MyDrive/thesis_v220_td3_runs'; os.makedirs(DRIVE_ROOT,exist_ok=True)
print('Drive mounted:',DRIVE_ROOT)


In [ ]:
# Clone repo + install deps (SB3 pinned 2.6.0). Same stack as the SAC/TD3 runs.
import subprocess, sys, os
WORK='/content'; repo=os.path.join(WORK,'thesis')
if os.path.exists(repo): subprocess.run(['rm','-rf',repo],check=True)
subprocess.run(['git','clone','https://github.com/taratorbati/thesis.git',repo],check=True)
os.chdir(repo); sys.path.insert(0,repo)
subprocess.run(['pip','install','--quiet','stable-baselines3==2.6.0','gymnasium','wandb','pytest'],check=True)
import torch; print(f'PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY'); print('OK WANDB key.')
except Exception as e:
    try:
        import getpass; k=getpass.getpass('WANDB key (Enter to skip): ').strip()
        if k: os.environ['WANDB_API_KEY']=k; print('OK set.')
        else: print('Skipping WandB.')
    except Exception: print('Skipping WandB.')
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'no GPU')


## [OPTIONAL] Write v2.21 files
Skip if already pushed. Writes the modified `gym_env.py` (adds the backward-compatible `biomass_shaping_gamma` param; v2.20 ignores it), `configs_v221.py`, and `train_v221_td3.py`. n-step components are unchanged in the repo.

In [ ]:
# [OPTIONAL] Write the v2.21 files into the cloned repo. Skip if already pushed.
from pathlib import Path
REPO = '/content/thesis'
_files = {
    'src/rl/configs_v221.py':
        '23207372632f726c2f636f6e666967735f763232312e7079202076322e32312e300a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d0a232076322e3231203d2076322e32302052756e204120286578616374206e2d737465702c'
        '20776869636820636f6e76657267656429202b204f4e45206368616e67653a207468652062696f6d6173730a232072657761'
        '726420723120737769746368657320746f207468652047414d4d412d434f525245435420706f74656e7469616c2d62617365'
        '642d73686170696e6720666f726d2e204f6e65207661726961626c652e0a230a2320723120746f646179202876322e323029'
        '3a2020202878345f74202d2078345f7b742d317d29202f2058345f52454620202020202020202d2d20696e6372656d656e74'
        '2c2074656c6573636f7065730a23202020202020202020202020202020202020202020746f207465726d696e616c20796965'
        '6c64204f4e4c592061742067616d6d613d312e0a2320723120696e2076322e32313a20202020202020202867616d6d61202a'
        '2078345f74202d2078345f7b742d317d29202f2058345f524546202d2d2074686520706f6c6963792d696e76617269616e74'
        '0a23202020202020202020202020202020202020202020706f74656e7469616c2d73686170696e6720666f726d20284e672c'
        '2048617261646120262052757373656c6c20313939392920776974680a232020202020202020202020202020202020202020'
        '20706f74656e7469616c20506869203d20414c504841312a78342f58345f5245462e0a230a23205748592074686973206973'
        '207468652065786163742066697820616e6420776879206974204d415443484553204d50430a23202d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a23204d504327732062696f'
        '6d617373206f626a65637469766520287372632f6d70632f636f73742e70792920697320612050555245205445524d494e41'
        '4c20284d6179657229207465726d3a0a2320202020204a5f62696f6d617373203d202d616c70686131202a2078345f746572'
        '6d696e616c202f2078345f7265662e0a23205468652076322e32302064656e736520696e6372656d656e7420776173206974'
        '732067616d6d613d31206571756976616c656e74202874656c6573636f70657320746f2078345f54202d2078345f30292c0a'
        '23206275742061742067616d6d613d302e39392069742073706c69747320696e746f2067616d6d615e542a78345f54202b20'
        '28312d67616d6d61292a73756d5f742067616d6d615e742a78345f74202d2d207468650a23207365636f6e64207465726d20'
        '66726f6e742d6c6f6164732067726f77746820616e6420737461727665732074686520726570726f64756374697665207068'
        '61736520287468652076322e32302052756e20410a232064726f75676874207365657361772c20776f727374206174206d6f'
        '6465726174652f373025292e2057697468207468652067616d6d612d636f727265637420666f726d2c207468652064697363'
        '6f756e7465640a232062696f6d6173732072657475726e2074656c6573636f7065732045584143544c593a0a232020202020'
        '73756d5f742067616d6d615e74202867616d6d612a78345f7b742b317d202d2078345f74292f58345f524546203d20286761'
        '6d6d615e542a78345f54202d2078345f30292f58345f5245462c0a2320692e652e20612050555245207465726d696e616c2d'
        '7969656c64206f626a6563746976652077697468204e4f2066726f6e742d6c6f6164696e67207465726d202d2d2070726f76'
        '61626c79207468650a232073616d65206f626a656374697665204d5043206f7074696d697365732c207768696c6520737461'
        '79696e672064656e73652f6c6561726e61626c652e2067616d6d6120697320756e746f75636865642c20736f0a2320746865'
        '20626f6f74737472617020686f72697a6f6e2028616e64207468652073746162696c697479206e2d7374657020626f756768'
        '742920697320756e6368616e6765642e0a230a23205468652073686170696e672067616d6d61206d75737420657175616c20'
        '746865207065722d737465702072657475726e20646973636f756e742e20576520646f204e4f5420686172642d636f646520'
        '69740a2320686572653b2074686520747261696e6572207365747320656e762062696f6d6173735f73686170696e675f6761'
        '6d6d61203d2067616d6d615f626173652028302e393929207768656e0a232062696f6d6173735f73686170696e673d547275'
        '652c20736f207468652074776f2063616e206e657665722064726966742e0a230a2320436f6d7061746962696c6974793a20'
        '6261636b776172642d636f6d70617469626c652028656e7620706172616d2064656661756c747320746f20312e30203d206f'
        '6c64206265686176696f75722c20736f0a23205341432f76322e3139622f76322e32302061726520627974652d6964656e74'
        '6963616c292e20546865206576616c206861726e65737320286578705f726c202d3e2072756e5f736561736f6e290a23206d'
        '6561737572657320504859534943414c206f7574636f6d657320616e64206e657665722075736573207468652067796d2072'
        '65776172642c20736f207969656c642f64726f756768742f77617465726c6f670a2320616e6420746865204d504320636f6d'
        '70617269736f6e2061726520756e6166666563746564202d2d2074686973206368616e676573204f4e4c5920686f77207468'
        '6520706f6c69637920697320747261696e65642e0a23204e6f74653a20747261696e696e6720726577617264202f20715f70'
        '726564205343414c4520736869667473202862696f6d6173732072657475726e207e67616d6d615e542a78345f542c20736d'
        '616c6c6572292c0a2320776869636820697320696e7465726e616c6c7920636f6e73697374656e74202d2d206a7564676520'
        '73746162696c69747920627920626f756e6465646e6573732c206e6f74206162736f6c7574652076616c75652e0a230a2320'
        '50756c73696e67202f204d61726b6f762d72352069732061207365706172617465206c617465722076657273696f6e20286e'
        '65656473206120392d66656174757265206e6574776f726b292e0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d0a66726f6d205f5f6675747572655f5f20696d706f727420616e6e6f746174696f6e730a0a47414d4d415f42'
        '415345203d20302e39390a0a52554e5f41203d2064696374280a202020206c6162656c3d22677368617065222c0a20202020'
        '23202d2d2d20696e686572697465642066726f6d2076322e32302052756e204120287468652065786163742d6e2d73746570'
        '2073746162696c697365722c20756e6368616e67656429202d2d2d0a202020206e5f73746570733d352c0a2020202067616d'
        '6d615f626173653d47414d4d415f424153452c0a202020206c6561726e696e675f7374617274733d35305f3030302c0a2020'
        '20207265776172645f64755f616c7068613d302e3030352c0a20202020706f6c6963795f64656c61793d322c0a2020202074'
        '61726765745f706f6c6963795f6e6f6973653d302e322c0a202020207461726765745f6e6f6973655f636c69703d302e352c'
        '0a202020206163746f725f6c725f6d756c743d312e302c0a202020206163746f725f7761726d75705f757064617465733d30'
        '2c0a202020206578706f73655f707265765f753d46616c73652c0a2020202023202d2d2d20746865204f4e4c59206368616e'
        '676520696e2076322e3231202d2d2d0a2020202062696f6d6173735f73686170696e673d547275652c202020232074726169'
        '6e6572207365747320656e762062696f6d6173735f73686170696e675f67616d6d61203d2067616d6d615f62617365202830'
        '2e3939290a290a0a434f4e46494753203d207b2241223a2052554e5f417d0a'
        ,
    'src/rl/train_v221_td3.py':
        '...'
        '2c0a20202020290a'
        ,
    'src/rl/gym_env.py':
        '...'
        ,
}
for relpath, hexstr in _files.items():
    p = Path(REPO) / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(bytes.fromhex(hexstr))
    print(f'  written: {relpath} ({p.stat().st_size:,} bytes)')
print('v2.21 files ready.')

In [ ]:
# Smoke tests + a v2.21 pilot that runs real gradient steps.
# Catches import errors and training-loop bugs before committing ~1 hr of GPU time.
import subprocess, sys
REPO = '/content/thesis'
print('Smoke tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_rl_smoke.py','-v','--tb=short'],cwd=REPO).returncode==0,'SMOKE FAILED'
print('\nFactorized-critic tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_factorized_critic.py','-v','--tb=short'],cwd=REPO).returncode==0,'CRITIC TESTS FAILED'
print('\nv2.21 pilot (runs real gradient steps with NStepReplayBufferExact)...')
from src.rl import configs_v221
import copy
_pilot = copy.deepcopy(configs_v221.CONFIGS['A'])
_pilot['learning_starts'] = 200  # small enough for 1200-step pilot
configs_v221.CONFIGS['PILOT'] = _pilot
from src.rl.train_v221_td3 import train_td3_v221
m = train_td3_v221('PILOT', seed=999, output_dir='/content/pilot', total_timesteps=1200)
import glob, zipfile, io, torch
ck = glob.glob('/content/pilot/td3_v221_gshape_seed999/*final*.zip')
if ck:
    with zipfile.ZipFile(ck[0]) as z:
        sd = torch.load(io.BytesIO(z.open('policy.pth').read()), map_location='cpu', weights_only=False)
    assert 'actor.log_std.weight' not in sd, 'BUG: log_std found (SAC checkpoint loaded?)'
    assert 'actor.mu_head.weight' in sd, 'BUG: mu_head missing'
    assert abs(float(sd['actor.obs_norm_marker'].item()) - 2.19) < 0.01, 'BUG: marker != 2.19'
    print('  Checkpoint: deterministic actor, mu_head present, marker=2.19. OK')
assert type(m.replay_buffer).__name__ == 'NStepReplayBufferExact', 'BUG: wrong buffer type'
assert m.replay_buffer.n_steps == 5, 'BUG: wrong n_steps'
print(f'  Buffer: {type(m.replay_buffer).__name__}  n_steps={m.replay_buffer.n_steps}  gamma={m.replay_buffer._n_gamma}. OK')
print(f'  model.gamma = {m.gamma:.6f}  (expect {0.99**5:.6f}). {"OK" if abs(m.gamma - 0.99**5)<1e-8 else "BUG"}')
print('\nOK pre-flight passed. Proceed to training.')


In [ ]:
# Full 250k TD3 v2.21 (gamma-correct biomass shaping). Inherits v2.20 Run A's exact
# n-step stabiliser; the ONLY change is r1 -> (gamma*x4_t - x4_{t-1})/X4_REF, which
# makes the biomass objective a pure terminal yield (== MPC). ~1-1.5 hr T4.
SEED = 0    # CHANGE per session
REPO = '/content/thesis'
from src.rl.train_v221_td3 import train_td3_v221
model = train_td3_v221(
    config_name='A',
    seed=SEED,
    output_dir=f'{REPO}/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    # ── params from CONFIGS['A'] in configs_v221.py ──
    # INHERITED FROM v2.20 RUN A (unchanged): n_steps=5, gamma_base=0.99,
    #   model.gamma=0.99**5, learning_starts=50_000, reward_du_alpha=0.005,
    #   policy_delay=2, target_policy_noise=0.2, actor warmup off.
    # THE ONLY v2.21 CHANGE:
    #   biomass_shaping=True -> trainer sets env biomass_shaping_gamma = gamma_base (0.99),
    #   so r1 = (0.99*x4_t - x4_{t-1})/X4_REF and the discounted biomass return
    #   telescopes to gamma^T*x4_T (pure terminal yield, == MPC's objective).
    #   expose_prev_u=False  (pulsing fix is a later version).
)
print('Training complete.')

In [ ]:
import shutil, os, datetime
RUN_LABEL = 'gshape'   # 'nstep5_damped' for Run B
src=f'/content/thesis/results/rl/td3_v221_{RUN_LABEL}_seed{SEED}'
dst=os.path.join(DRIVE_ROOT,f'td3_v221_{RUN_LABEL}_seed{SEED}_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree(src,dst,ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to Drive:',dst)


In [ ]:
# Post-training eval: best_model + final model, perfect + noisy forecasts.
# Writes parquets to results/runs/<tag>/ for the comparison diagnostic.
import subprocess, sys, os
REPO = '/content/thesis'
RUN_LABEL = 'gshape'   # match what you ran
run_dir = f'{REPO}/results/rl/td3_v221_{RUN_LABEL}_seed{SEED}'
model_path = f'{run_dir}/best_model/best_model.zip'
final_path = f'{run_dir}/td3_v221_{RUN_LABEL}_seed{SEED}_final.zip'
BEST_TAG=f'td3_v221_{RUN_LABEL}_best_seed{SEED}'; FINAL_TAG=f'td3_v221_{RUN_LABEL}_final_seed{SEED}'
print('Evaluating BEST (perfect)...')
r=subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','perfect',
    '--force','--out-tag',BEST_TAG],capture_output=True,text=True,cwd=REPO)
print(r.stdout[-1500:])
if r.returncode!=0: print('STDERR:',r.stderr[-2500:])
assert r.returncode==0,'PERFECT EVAL FAILED'
print('\nEvaluating BEST (noisy, forecast-sensitivity check)...')
subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','noisy',
    '--noise-seed','42','--force','--out-tag',BEST_TAG],cwd=REPO)
if os.path.exists(final_path):
    print('\nEvaluating FINAL (250k, perfect)...')
    subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
        '--model',final_path,'--scenario','all','--budget','all','--forecast','perfect',
        '--force','--out-tag',FINAL_TAG],cwd=REPO)
print(f'\nBest-model eval -> results/runs/{BEST_TAG}/')


In [ ]:
# PRIMARY v2.21 DIAGNOSTIC: did the drought seesaw drop while yield held? (vs MPC)
# The gamma-correct shaping makes the objective pure-terminal-yield, so the agent
# should protect the reproductive phase -> fewer drought-days, yield held/up.
# Re-check stability next cell (NOTE: reward SCALE shifts vs v2.20, so q_pred will be
# numerically different -- judge by boundedness/recovery, not the absolute value).
import pandas as pd, numpy as np, json, glob, os
REPO = '/content/thesis'; RUN_LABEL = 'gshape'
OUT=f'{REPO}/results/runs/td3_v221_{RUN_LABEL}_best_seed{SEED}'
MPC=f'{REPO}/results/runs'
assert os.path.isdir(OUT), f'{OUT} missing -- run the eval cell first.'
def load_m(d, scen, b, kind):
    g = (glob.glob(os.path.join(d,f'sac_perfect_det_{scen}_rice_{b}pct_seed*.json')) if kind=='v221'
         else glob.glob(os.path.join(d,f'mpc_perfect_{scen}_rice_{b}pct_Hp8.json')))
    return json.load(open(g[0]))['final_metrics'] if g else None
print('='*80)
print(f'{"cell":<13}{"v221 yld":>9}{"MPC yld":>8}{"d":>6} | {"v221 drght":>11}{"MPC drght":>10} | {"v221 wlog":>10}')
yA,yM,dA,dM,wA=[],[],[],[],[]
for scen in ['dry','moderate','wet']:
    for b in ['100','85','70']:
        a=load_m(OUT,scen,b,'v221'); m=load_m(MPC,scen,b,'mpc')
        if a and m:
            yA.append(a['yield_kg_ha']); yM.append(m['yield_kg_ha'])
            dA.append(a['drought_days_per_agent']); dM.append(m['drought_days_per_agent']); wA.append(a['waterlog_days_per_agent'])
            print(f'{scen+"/"+b+"%":<13}{a["yield_kg_ha"]:>9.0f}{m["yield_kg_ha"]:>8.0f}'
                  f'{a["yield_kg_ha"]-m["yield_kg_ha"]:>+6.0f} | '
                  f'{a["drought_days_per_agent"]:>11.1f}{m["drought_days_per_agent"]:>10.1f} | '
                  f'{a["waterlog_days_per_agent"]:>10.1f}')
print('='*80)
ay,my=np.mean(yA),np.mean(yM); ad,md=np.mean(dA),np.mean(dM)
print(f'9-CELL MEAN  yield: v221={ay:.0f} ({100*ay/my:.1f}% MPC; v2.20 RunA=99.8%)   '
      f'drought-d: v221={ad:.1f}  MPC={md:.1f}  (v2.20 RunA~33; target <=22)')
print(f'             mod/70% yield: v221={[load_m(OUT,"moderate","70","v221")][0]["yield_kg_ha"]:.0f} '
      f'vs MPC {[load_m(MPC,"moderate","70","mpc")][0]["yield_kg_ha"]:.0f}  (v2.20 RunA was -150)')
print('GATE: drought down toward MPC AND yield >=99.5% MPC AND wet waterlog still <= MPC.')

In [ ]:
# STABILITY DIAGNOSTIC. q_pred calibration + collapse guard + coverage.
# Stage-1 success = q_pred BOUNDED (not monotone), guard never trips.
import os, glob, numpy as np, pandas as pd
REPO = '/content/thesis'
RUN_LABEL = 'gshape'
run_dir=f'{REPO}/results/rl/td3_v221_{RUN_LABEL}_seed{SEED}'
br=os.path.join(run_dir,'bias_ratio_log.csv')
if os.path.exists(br):
    b=pd.read_csv(br); print('--- bias_ratio_log (q_pred trajectory) ---')
    print(b.to_string(index=False))
    q_min=float(b['q_pred_mean'].min()); q_final=float(b['q_pred_mean'].iloc[-1])
    q_mono = all(b['q_pred_mean'].diff().dropna() < 0)   # True if always decreasing
    print(f'\n  q_pred min={q_min:+.1f}  final={q_final:+.1f}  monotone_dive={q_mono}')
    if q_mono:
        verdict = 'FAIL -- monotone dive (same as v2.20 r5). Try Run B.'
    elif q_min < -30 and q_final >= -5:
        verdict = 'PARTIAL -- deep dip but recovered. Watch over seeds; consider Run B.'
    elif q_final >= -5:
        verdict = 'PASS -- q_pred bounded and recovered.'
    else:
        verdict = f'UNCERTAIN -- q_pred final={q_final:+.1f}. Check the curve.'
    print(f'  VERDICT: {verdict}')
else:
    print('No bias_ratio_log.csv at', br)

cg=os.path.join(run_dir,'collapse_guard_log.csv')
if os.path.exists(cg):
    g=pd.read_csv(cg)
    tripped=int(g['collapsed'].max()) if 'collapsed' in g.columns and len(g) else 0
    last_frac=g['frac_low_rolling'].iloc[-1] if len(g) else float('nan')
    print(f'\n--- collapse_guard ---  rows={len(g)}  final_frac_low={last_frac:.0%}')
    print(f'  guard tripped: {"YES -- collapsed" if tripped else "NO -- healthy"}')

cov=os.path.join(run_dir,'low_action_coverage_log.csv')
if os.path.exists(cov):
    c=pd.read_csv(cov)
    print(f'--- low_action_coverage ---  mean frac_low (last 50k)={c["frac_low_action"].tail(50).mean():.0%}')

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    runs=glob.glob(os.path.join(run_dir,'tensorboard','*'))
    if runs:
        ea=EventAccumulator(runs[0]); ea.Reload()
        if 'train/critic_loss' in ea.Tags()['scalars']:
            cl=ea.Scalars('train/critic_loss'); mxl=max(e.value for e in cl)
            print(f'\n  max critic_loss = {mxl:.2f}  (STABLE if < 100; v2.7 cascade hit 6.9e12)')
except Exception as e:
    print('tensorboard read skipped:', e)


## [NEXT VERSION] Pulsing / Markov-r5 (needs a network change)

v2.21 targets **drought** via the gamma-correct biomass objective. The other gap — **pulsing** (mean|Δu| 2.04 vs MPC 0.98) — needs r5 made *Markov* (`expose_prev_u=True`), which requires a 9-feature actor+critic first (`networks_td3.py` hard-codes 8). Make it its own version (v2.22).

In [ ]:
# [OPTIONAL] Resume v2.21 from a checkpoint (after a Colab disconnect).
# from src.rl.train_v221_td3 import WarmupAsymmetricLRTD3
# SEED=0; RUN_LABEL='gshape'; STEP=150_000
# ckpt=f'/content/thesis/results/rl/td3_v221_{RUN_LABEL}_seed{SEED}/checkpoints/td3_v221_{RUN_LABEL}_seed{SEED}_{STEP}_steps.zip'